# 23N Field Tests #01 — The Solar Dust Tax

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/main/ft01-solar-dust-tax/ft01_solar_dust_tax.ipynb)

**Question.** How much revenue does dust (soiling) cost a large solar park in Kutch, Gujarat — and how much does
cleaning more often win back? We (a) fine-tune a small CNN to read **% power loss from a photo of a panel**,
(b) pull a full year of **daily sunlight and rain for Khavda, Kutch** from NASA POWER, and (c) combine them in a
transparent yield + soiling model for a **1 GW** park, comparing cleaning every 7, 14 and 30 days.

**Data.**
- *DeepSolarEye* (Mehta et al., WACV 2018): 45,754 RGB images of a soiled panel, each labelled with % power loss
  versus an identical clean reference panel. Licence: **CC BY-NC-SA 2.0** (as stated on the project page,
  deep-solar-eye.github.io). Fetched here from a public GitHub mirror (pinned commit) because the original is a
  Google Drive folder.
- *NASA POWER* daily point API (`ALLSKY_SFC_SW_DWN`, `PRECTOTCORR`): free, no key; NASA asks for acknowledgement.

**Runtime.** ~10–14 min on free Colab with a T4 GPU (Runtime → Change runtime type → T4). ~20 min on CPU (the
notebook automatically trains on a subsample). No paid API keys.

*Independent study on public data by 23N AI Labs. Not affiliated with, or endorsed by, any park developer,
operator, utility or government body. "Khavda" is used only as a location. All money figures are estimates.*

**Sections:** 1 Setup → 2 Data → 3 Model & method → 4 Results (headline) → 5 Interactive demo → 6 Video render →
7 Post text → 8 Limitations

## 1. Setup

In [ ]:
%pip install -q gradio

In [ ]:
# ==== CONFIG: every assumption is a named variable. Edit and re-run. ====
import os, re, io, json, math, time, glob, random, subprocess, shutil, datetime as dt
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from PIL import Image

FT_NUM = "01"
FT_TITLE = "The Solar Dust Tax"

# --- Site (Khavda, Kutch, Gujarat; NASA POWER solar grid is ~0.5 deg, so +/- 0.1 deg does not matter)
SITE_NAME = "Khavda, Kutch, Gujarat"
LAT, LON = 23.90, 69.70
YEAR = dt.date.today().year - 1          # last full calendar year

# --- Park & money (estimates, see README for sources)
PARK_MW = 1000                           # 1 GW (DC) park
PERFORMANCE_RATIO = 0.80                 # all losses EXCEPT soiling (temperature, inverter, wiring, availability)
TARIFF_RS_PER_KWH = 2.60                 # GUVNL solar auctions: Rs 2.60 (Mar 2025), Rs 2.34-2.60 (Phase XXVIII, Mar 2026)
CRORE = 1e7

# --- Soiling model (Kimber-style: linear build-up, heavy rain or a wash resets to zero)
SOILING_RATE_PCT_PER_DAY = 0.37          # Valerino et al., Solar Energy 204 (2020): 0.37 +/- 0.09 %/day, Gandhinagar
RAIN_CLEAN_MM = 6.0                      # daily rain that cleans panels (pvlib.soiling.kimber default)
GRACE_DAYS = 14                          # damp-ground days after rain with no new dust (pvlib kimber default)
MAX_SOILING_PCT = 30.0                   # ceiling on soiling loss (pvlib kimber default)
CLEANING_INTERVALS = [7, 14, 30]         # days between manual washes to compare
HEADLINE_INTERVAL = 7                    # interval quoted in the headline

# --- CV model
SEED = 23
PRETRAINED = True                        # ImageNet weights from torchvision (downloaded once, ~45 MB)
TEST_DAY_START, TEST_DAY_STRIDE = 2, 5   # hold out every 5th capture day (3 of 15) — whole days, so no leakage between 5-s frames
DATA_DIR = "deepsolareye"
DATA_REPO = "https://github.com/ThomasGust/SoilingNet.git"   # public mirror of DeepSolarEye 'PanelImages'
DATA_COMMIT = "87c4ed6b15db011f61a3b61f53236a4f973888fd"
EXPECTED_IMAGES = 45754

import torch, torch.nn as nn, torchvision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    IMG_SIZE, EPOCHS, BATCH, MAX_TRAIN, MAX_TEST = 128, 6, 256, None, None
else:   # CPU fallback: subsample so Run-all stays well under 25 min
    IMG_SIZE, EPOCHS, BATCH, MAX_TRAIN, MAX_TEST = 96, 2, 64, 6000, 2000

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"torch {torch.__version__} | device={DEVICE} | img={IMG_SIZE} epochs={EPOCHS} "
      f"train_cap={MAX_TRAIN} test_cap={MAX_TEST}")

## 2. Data
### 2a. DeepSolarEye panel images (power-loss labels)

Each file name encodes the capture time, the **power loss `L`** (fraction, vs. a clean reference panel) and the
**irradiance `I`** (normalised), e.g. `solar_Fri_Jun_16_10__0__11_2017_L_0.906153208302_I_0.321592156863.jpg`.
We fetch only the `PanelImages/` folder (~45k JPEGs, 192×192) with a sparse git checkout of a pinned commit (~30–60 s).
The original release is a Google Drive folder linked from deep-solar-eye.github.io; if the mirror ever disappears,
download it from there (e.g. with `gdown --folder`) and point `DATA_DIR` at the folder containing `PanelImages/`.

In [ ]:
def fetch_deepsolareye(root=DATA_DIR):
    img_dir = os.path.join(root, "PanelImages")
    if os.path.isdir(img_dir) and len(os.listdir(img_dir)) >= EXPECTED_IMAGES:
        return img_dir
    t0 = time.time()
    shutil.rmtree(root, ignore_errors=True)
    cmds = [["git", "init", "-q", root],
            ["git", "-C", root, "remote", "add", "origin", DATA_REPO],
            ["git", "-C", root, "sparse-checkout", "set", "--no-cone", "PanelImages/"],
            ["git", "-C", root, "fetch", "-q", "--depth", "1", "--filter=blob:none", "origin", DATA_COMMIT],
            ["git", "-C", root, "checkout", "-q", "FETCH_HEAD"]]
    for c in cmds:
        subprocess.run(c, check=True)
    print(f"Fetched {len(os.listdir(img_dir)):,} images in {time.time()-t0:.0f}s")
    return img_dir

IMG_DIR = fetch_deepsolareye()

NAME_RE = re.compile(r"solar_\w{3}_(\w{3})_(\d+)_(\d+)__(\d+)__(\d+)_(\d{4})_L_([0-9.eE-]+)_I_([0-9.eE-]+?)\.jpg")
rows = []
for f in os.listdir(IMG_DIR):
    m = NAME_RE.match(f)
    if m:
        mon, day, hh, mm, ss, yr, L, I = m.groups()
        ts = pd.Timestamp(f"{yr}-{mon}-{int(day):02d} {int(hh):02d}:{int(mm):02d}:{int(ss):02d}")
        rows.append((f, ts, float(L), float(I)))
meta = pd.DataFrame(rows, columns=["file", "ts", "loss", "irr"]).sort_values("ts").reset_index(drop=True)
meta["day"] = meta.ts.dt.date
print(f"{len(meta):,} labelled images over {meta.day.nunique()} days "
      f"({meta.day.min()} → {meta.day.max()}); unparsed files: {len(os.listdir(IMG_DIR)) - len(meta)}")
print(meta.loss.describe(percentiles=[.1, .5, .9]).round(3).to_string())

In [ ]:
# Split by whole DAY (consecutive frames 5 s apart are near-duplicates, so a random split would leak).
days = sorted(meta.day.unique())
test_days = set(days[TEST_DAY_START::TEST_DAY_STRIDE])
train_df = meta[~meta.day.isin(test_days)]
test_df = meta[meta.day.isin(test_days)]
if MAX_TRAIN and len(train_df) > MAX_TRAIN: train_df = train_df.sample(MAX_TRAIN, random_state=SEED)
if MAX_TEST and len(test_df) > MAX_TEST:   test_df = test_df.sample(MAX_TEST, random_state=SEED)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)
print(f"train {len(train_df):,} images ({len(days)-len(test_days)} days) | test {len(test_df):,} images "
      f"(held-out days: {', '.join(str(d) for d in sorted(test_days))})")

from concurrent.futures import ThreadPoolExecutor
def load_img(path, size=IMG_SIZE):
    with Image.open(path) as im:
        im.draft("RGB", (size, size))
        return np.asarray(im.convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8)

def load_many(files, size=IMG_SIZE):
    with ThreadPoolExecutor(max_workers=8) as ex:
        return np.stack(list(ex.map(lambda f: load_img(os.path.join(IMG_DIR, f), size), files)))

t0 = time.time()
X_train, y_train = load_many(train_df.file), train_df.loss.values.astype(np.float32)
X_test, y_test = load_many(test_df.file), test_df.loss.values.astype(np.float32)
print(f"decoded {len(X_train)+len(X_test):,} images in {time.time()-t0:.0f}s → "
      f"{(X_train.nbytes+X_test.nbytes)/1e9:.2f} GB in RAM")

In [ ]:
# Quick look: clean → heavily soiled, with measured power loss
qs = [0.0, 0.2, 0.4, 0.6, 0.8, 0.95]
picks = [meta.iloc[(meta.loss - meta.loss.quantile(q)).abs().argsort().iloc[0]] for q in qs]
fig, axs = plt.subplots(1, len(picks) + 1, figsize=(18, 3.2), gridspec_kw=dict(width_ratios=[1]*len(picks) + [1.6]))
for ax, r in zip(axs, picks):
    ax.imshow(Image.open(os.path.join(IMG_DIR, r.file))); ax.set_title(f"loss {r.loss*100:.0f}%"); ax.axis("off")
axs[-1].hist(meta.loss * 100, bins=40, color="#FF7A1A"); axs[-1].set_xlabel("measured power loss (%)")
axs[-1].set_title("label distribution"); plt.tight_layout(); plt.show()

### 2b. NASA POWER — daily sunlight and rain at Khavda, Kutch

`ALLSKY_SFC_SW_DWN` = all-sky surface shortwave (global horizontal) irradiance, daily total;
`PRECTOTCORR` = bias-corrected precipitation (mm/day). Free API, no key:
`https://power.larc.nasa.gov/api/temporal/daily/point?...`

In [ ]:
import requests

def fetch_nasa_power(lat, lon, year, params=("ALLSKY_SFC_SW_DWN", "PRECTOTCORR")):
    url = ("https://power.larc.nasa.gov/api/temporal/daily/point"
           f"?parameters={','.join(params)}&community=RE&longitude={lon}&latitude={lat}"
           f"&start={year}0101&end={year}1231&format=JSON")
    r = requests.get(url, timeout=120); r.raise_for_status()
    js = r.json()
    fill = js.get("header", {}).get("fill_value", -999)
    units = {p: js.get("parameters", {}).get(p, {}).get("units", "?") for p in params}
    df = pd.DataFrame({p: js["properties"]["parameter"][p] for p in params})
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df = df.replace(fill, np.nan).astype(float)
    # The RE community returns kWh/m^2/day for irradiance; convert defensively if MJ is ever returned.
    if "MJ" in units.get("ALLSKY_SFC_SW_DWN", ""):
        df["ALLSKY_SFC_SW_DWN"] /= 3.6
    return df, units, url

In [ ]:
wx, wx_units, wx_url = fetch_nasa_power(LAT, LON, YEAR)
n_missing = int(wx.ALLSKY_SFC_SW_DWN.isna().sum())
wx["ALLSKY_SFC_SW_DWN"] = wx.ALLSKY_SFC_SW_DWN.interpolate(limit_direction="both")
wx["PRECTOTCORR"] = wx.PRECTOTCORR.fillna(0.0)
GHI = wx.ALLSKY_SFC_SW_DWN.values           # kWh/m^2/day  (= peak-sun-hours)
RAIN = wx.PRECTOTCORR.values                 # mm/day
GHI_AVG = float(np.mean(GHI))
print(f"NASA POWER {SITE_NAME} ({LAT}N, {LON}E), {YEAR}: {len(wx)} days, units={wx_units}, "
      f"missing irradiance days filled={n_missing}")
print(f"mean GHI {GHI_AVG:.2f} kWh/m²/day | annual {GHI.sum():,.0f} kWh/m² | rain {RAIN.sum():.0f} mm | "
      f"days with rain ≥ {RAIN_CLEAN_MM} mm: {(RAIN >= RAIN_CLEAN_MM).sum()}")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(wx.index, GHI, color="#2EC4B6", lw=0.8, alpha=0.6, label="daily GHI")
ax.plot(wx.index, pd.Series(GHI, index=wx.index).rolling(7, center=True, min_periods=1).mean(), color="#FF7A1A", lw=2, label="7-day mean")
ax.set_ylabel("kWh/m²/day"); ax2 = ax.twinx(); ax2.bar(wx.index, RAIN, color="tab:blue", alpha=0.4, label="rain"); ax2.set_ylabel("rain (mm/day)")
ax.set_title(f"NASA POWER daily irradiance & rain — {SITE_NAME}, {YEAR}"); ax.legend(loc="lower left"); plt.show()

## 3. Model & method
### 3a. CNN: photo → % power loss

ResNet18 (ImageNet-pretrained, torchvision) with a single sigmoid output, trained with an L1 loss so the metric we
report — **mean absolute error in percentage points** — is what we optimise. Images are pre-loaded into RAM and
augmented on the GPU (flips, brightness) so a T4 finishes in a few minutes.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def to_tensor(xb, train=False):
    x = torch.from_numpy(xb).to(DEVICE).permute(0, 3, 1, 2).float() / 255.0
    if train:
        flip = torch.rand(x.size(0), device=DEVICE) < 0.5
        x[flip] = x[flip].flip(3)
        x = (x * (0.8 + 0.4 * torch.rand(x.size(0), 1, 1, 1, device=DEVICE))).clamp(0, 1)
    return (x - MEAN) / STD

def build_model(pretrained=PRETRAINED):
    w = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
    m = torchvision.models.resnet18(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return m.to(DEVICE)

@torch.no_grad()
def predict_loss(model, X, bs=512):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            out.append(torch.sigmoid(model(to_tensor(X[i:i+bs]))).float().squeeze(1).cpu().numpy())
    return np.concatenate(out)

model = build_model()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
steps_per_epoch = math.ceil(len(X_train) / BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * steps_per_epoch, pct_start=0.2)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
y_train_t = torch.from_numpy(y_train)

t0 = time.time()
for ep in range(EPOCHS):
    model.train(); perm = np.random.permutation(len(X_train)); run = 0.0
    for i in range(steps_per_epoch):
        idx = perm[i*BATCH:(i+1)*BATCH]
        xb, yb = to_tensor(X_train[idx], train=True), y_train_t[idx].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            pred = torch.sigmoid(model(xb).squeeze(1).float())
            loss = nn.functional.l1_loss(pred, yb)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
    p = predict_loss(model, X_test)
    print(f"epoch {ep+1}/{EPOCHS}  train L1 {run/steps_per_epoch*100:.2f} pts  |  "
          f"held-out MAE {np.abs(p - y_test).mean()*100:.2f} pts  |  {time.time()-t0:.0f}s")

In [ ]:
pred_test = predict_loss(model, X_test)
MAE_PTS = float(np.abs(pred_test - y_test).mean() * 100)
BASELINE_MAE_PTS = float(np.abs(np.median(y_train) - y_test).mean() * 100)
N_IMAGES, N_TRAIN = len(meta), len(train_df)
print(f"Held-out-day MAE: {MAE_PTS:.1f} percentage points  "
      f"(baseline 'always predict the median': {BASELINE_MAE_PTS:.1f} pts)")
for lo, hi in [(0, .1), (.1, .3), (.3, .6), (.6, 1.01)]:
    s = (y_test >= lo) & (y_test < hi)
    if s.any(): print(f"  true loss {lo*100:>3.0f}-{min(hi,1)*100:>3.0f}%: n={s.sum():>5}, MAE {np.abs(pred_test[s]-y_test[s]).mean()*100:.1f} pts")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test * 100, pred_test * 100, s=3, alpha=0.3, color="#2EC4B6"); ax.plot([0, 100], [0, 100], "--", color="#FF7A1A")
ax.set_xlabel("measured loss (%)"); ax.set_ylabel("predicted loss (%)"); ax.set_title(f"held-out days · MAE {MAE_PTS:.1f} pts"); plt.show()

# Showcase samples (well-lit frames spanning clean → dirty) for the demo & video
lit = test_df.irr.values > 0.2
SAMPLES = []
for target in [0.02, 0.25, 0.55, 0.85]:
    cand = np.where(lit)[0] if lit.any() else np.arange(len(test_df))
    j = cand[np.argmin(np.abs(y_test[cand] - target))]
    SAMPLES.append(dict(file=test_df.file[j], img=np.asarray(Image.open(os.path.join(IMG_DIR, test_df.file[j])).convert("RGB")),
                        true=float(y_test[j] * 100), pred=float(pred_test[j] * 100)))
os.makedirs("ft01_examples", exist_ok=True)
EXAMPLE_PATHS = []
for k, s in enumerate(SAMPLES):
    pth = f"ft01_examples/panel_{k}_loss{s['true']:.0f}.jpg"; Image.fromarray(s["img"]).save(pth); EXAMPLE_PATHS.append(pth)
print([(round(s["true"]), round(s["pred"])) for s in SAMPLES])

### 3b. Yield + soiling model for a 1 GW park

* **Energy (no soiling):** `E_day = PARK_MW × 1000 kWp × GHI_day (kWh/m²/day ÷ 1 kW/m²) × PERFORMANCE_RATIO`.
  Using horizontal irradiance is conservative — tilted or tracking arrays collect more.
* **Soiling (Kimber-style, as in `pvlib.soiling.kimber`):** loss grows by `SOILING_RATE_PCT_PER_DAY` each dry day,
  capped at `MAX_SOILING_PCT`; a day with ≥ `RAIN_CLEAN_MM` of rain washes panels and the wet ground gives
  `GRACE_DAYS` without new dust; a scheduled manual wash every *N* days resets loss to zero. The year is simulated
  twice so 1 January starts from a realistic (steady-state) dust level.
* **Money:** `₹ lost = Σ E_day × soiling_day × TARIFF_RS_PER_KWH`.

In [ ]:
def simulate_soiling(rain_mm, interval_days=None, rate_pct=SOILING_RATE_PCT_PER_DAY, cap_pct=MAX_SOILING_PCT,
                     rain_mm_clean=RAIN_CLEAN_MM, grace_days=GRACE_DAYS):
    """Daily soiling loss fraction. interval_days=None → no manual washing (rain only)."""
    rate, cap = rate_pct / 100, cap_pct / 100
    def run(level, grace):
        out = np.zeros(len(rain_mm))
        for d, r in enumerate(rain_mm):
            if r >= rain_mm_clean: level, grace = 0.0, grace_days
            if interval_days and d % interval_days == 0: level = 0.0
            out[d] = level
            if grace > 0: grace -= 1
            else: level = min(level + rate, cap)
        return out, level, grace
    _, level, grace = run(0.0, 0)          # spin-up pass
    return run(level, grace)[0]

E_DAY_KWH = PARK_MW * 1000 * GHI * PERFORMANCE_RATIO
E_YEAR_KWH = float(E_DAY_KWH.sum())
SPEC_YIELD_KWH_PER_MW = E_YEAR_KWH / PARK_MW
REVENUE_CR = E_YEAR_KWH * TARIFF_RS_PER_KWH / CRORE

def scenario(interval, **kw):
    s = simulate_soiling(RAIN, interval, **kw)
    lost_kwh = float((E_DAY_KWH * s).sum())
    return dict(s=s, avg_loss_pct=float(s.mean() * 100), energy_weighted_loss_pct=lost_kwh / E_YEAR_KWH * 100,
                lost_gwh=lost_kwh / 1e6, lost_cr=lost_kwh * TARIFF_RS_PER_KWH / CRORE,
                washes=0 if not interval else int(np.ceil(len(RAIN) / interval)))

SCEN = {None: scenario(None)}
for n in CLEANING_INTERVALS: SCEN[n] = scenario(n)
print(f"Clean-panel yield: {E_YEAR_KWH/1e6:,.0f} GWh/yr ({SPEC_YIELD_KWH_PER_MW/1000:,.0f} kWh/kWp) → "
      f"revenue ₹{REVENUE_CR:,.0f} crore/yr at ₹{TARIFF_RS_PER_KWH:.2f}/kWh")

## 4. Results — the headline

In [ ]:
base = SCEN[None]
table = pd.DataFrame([dict(scenario="no washing (rain only)" if n is None else f"wash every {n} days",
                           washes_per_year=v["washes"], avg_soiling_loss_pct=round(v["energy_weighted_loss_pct"], 2),
                           energy_lost_GWh=round(v["lost_gwh"], 1), rs_crore_lost=round(v["lost_cr"], 1),
                           rs_crore_recovered_vs_no_washing=round(base["lost_cr"] - v["lost_cr"], 1))
                      for n, v in SCEN.items()])
print(table.to_string(index=False))

LOSS_UNCLEANED_CR = base["lost_cr"]
RECOVERED_CR = base["lost_cr"] - SCEN[HEADLINE_INTERVAL]["lost_cr"]
SLOW = max(CLEANING_INTERVALS)
extra_washes = SCEN[HEADLINE_INTERVAL]["washes"] - SCEN[SLOW]["washes"]
BREAKEVEN_RS_PER_MW = (SCEN[SLOW]["lost_cr"] - SCEN[HEADLINE_INTERVAL]["lost_cr"]) * CRORE / max(extra_washes, 1) / PARK_MW

HEADLINE = (f"Dust costs a 1 GW solar park in Kutch ~₹{LOSS_UNCLEANED_CR:,.0f} crore a year if left uncleaned; "
            f"cleaning every {HEADLINE_INTERVAL} days recovers ~₹{RECOVERED_CR:,.0f} crore.")
print("\nHEADLINE (estimate):", HEADLINE)
print(f"Going from every {SLOW} to every {HEADLINE_INTERVAL} days adds {extra_washes} washes/yr and pays off if a wash costs "
      f"< ₹{BREAKEVEN_RS_PER_MW:,.0f} per MW per wash.")

# Sensitivity of the headline to the two most uncertain inputs
rows = []
for rate in [0.2, SOILING_RATE_PCT_PER_DAY, 0.5]:
    for tariff in [2.34, TARIFF_RS_PER_KWH, 2.70]:
        sb, sh = simulate_soiling(RAIN, None, rate_pct=rate), simulate_soiling(RAIN, HEADLINE_INTERVAL, rate_pct=rate)
        lb, lh = (E_DAY_KWH * sb).sum() * tariff / CRORE, (E_DAY_KWH * sh).sum() * tariff / CRORE
        rows.append(dict(rate_pct_day=rate, tariff=tariff, uncleaned_cr=round(lb), recovered_cr=round(lb - lh)))
print("\nSensitivity (₹ crore/yr):"); print(pd.DataFrame(rows).pivot(index="rate_pct_day", columns="tariff", values="recovered_cr")
                                          .rename_axis("recovered ₹cr: rate %/day ↓  tariff ₹/kWh →").to_string())

fig, ax = plt.subplots(figsize=(12, 4))
for n, c in [(None, "grey"), (SLOW, "#FF7A1A"), (HEADLINE_INTERVAL, "#2EC4B6")]:
    ax.plot(wx.index, SCEN[n]["s"] * 100, color=c, lw=1.5, label="rain only" if n is None else f"wash every {n} d")
ax.set_ylabel("soiling loss (%)"); ax.set_title(f"Simulated soiling loss, {SITE_NAME}, {YEAR}"); ax.legend(); plt.show()

## 5. Interactive demo (Gradio)
Upload a photo of a panel → predicted % power loss → ₹ lost per MW per year *if that level of soiling persisted all
year* at this site. The model was trained on one rooftop rig (DeepSolarEye), so photos framed like the examples work
best. A public `*.gradio.live` link is printed below; Run-all does not block on it.

In [ ]:
import gradio as gr

def _square(img, size=IMG_SIZE):
    im = Image.fromarray(img).convert("RGB"); w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return np.asarray(im.resize((size, size), Image.BILINEAR), dtype=np.uint8)

def demo_predict(img):
    if img is None:
        return "Upload a solar-panel photo (or click an example)."
    loss = float(predict_loss(model, _square(img)[None])[0])
    rs_per_mw = loss * SPEC_YIELD_KWH_PER_MW * TARIFF_RS_PER_KWH
    days = loss * 100 / SOILING_RATE_PCT_PER_DAY
    return (f"## Predicted power loss: **{loss*100:.1f}%**\n"
            f"- ≈ **₹{rs_per_mw/1e5:,.1f} lakh per MW per year** if it stayed this dirty all year at {SITE_NAME} "
            f"({SPEC_YIELD_KWH_PER_MW/1000:,.0f} kWh/kWp/yr × ₹{TARIFF_RS_PER_KWH:.2f}/kWh)\n"
            f"- ≈ **₹{rs_per_mw*PARK_MW/CRORE:,.0f} crore/yr** for a {PARK_MW/1000:g} GW park\n"
            f"- ≈ {days:,.0f} days of dust at {SOILING_RATE_PCT_PER_DAY}%/day\n\n"
            f"*Estimate. Model MAE on held-out days: {MAE_PTS:.1f} pts. Independent study on public data.*")

demo = gr.Interface(fn=demo_predict, inputs=gr.Image(type="numpy", label="Solar panel photo"),
                    outputs=gr.Markdown(), examples=EXAMPLE_PATHS,
                    title="23N Field Tests #01 — The Solar Dust Tax",
                    description="Photo → % power loss (ResNet18 on DeepSolarEye) → ₹ per MW per year (NASA POWER sunlight, Kutch).")
print(demo_predict(SAMPLES[-1]["img"]))
try:
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio launch skipped:", e)

## 6. Video render (1080×1080, 30 fps, ~58 s, H.264) + cover (1080×1350)
Frames are drawn with matplotlib and piped straight into ffmpeg. Captions are burned in for muted autoplay.

In [ ]:
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import textwrap as _tw

BG, FG, ACC, TEAL, MUTED, PANEL = "#0B0F14", "#F5F7FA", "#FF7A1A", "#2EC4B6", "#8A94A6", "#161C24"
FPS, VW, VH = 30, 1080, 1080
matplotlib.rcParams["font.family"] = "DejaVu Sans"

def _ease(x):
    x = min(max(x, 0.0), 1.0); return 1 - (1 - x) ** 3

def _fig(w=VW, h=VH):
    f = Figure(figsize=(w / 100, h / 100), dpi=100, facecolor=BG); FigureCanvasAgg(f); return f

def _chrome(f, ctx, caption=None):
    f.text(0.045, 0.955, f"23N FIELD TESTS · #{ctx['ft_num']}", color=ACC, fontsize=17, fontweight="bold", va="top")
    f.text(0.955, 0.955, ctx["title"].upper(), color=MUTED, fontsize=15, va="top", ha="right")
    if caption:
        f.text(0.5, 0.06, "\n".join(_tw.wrap(caption, 62)), ha="center", va="center", color=FG, fontsize=21,
               linespacing=1.35, bbox=dict(boxstyle="round,pad=0.6", fc=PANEL, ec="none"))

def _ax_style(ax):
    ax.set_facecolor(BG)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]: ax.spines[s].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelsize=14)

def _scene_hook(f, t, ctx):
    a = _ease(t / 0.8)
    f.text(0.5, 0.60, "\n".join(_tw.wrap(ctx["hook"], 20)), ha="center", va="center", color=FG, fontsize=64,
           fontweight="bold", alpha=a, linespacing=1.15)
    f.text(0.5, 0.33, ctx["hook_sub"], ha="center", color=ACC, fontsize=26, alpha=_ease((t - 0.6) / 0.8))
    _chrome(f, ctx, "An independent study on public data")

def _scene_panels(f, t, ctx):
    f.text(0.5, 0.885, "A small CNN reads power loss from a photo", ha="center", color=FG, fontsize=32, fontweight="bold")
    pos = [(0.07, 0.50), (0.53, 0.50), (0.07, 0.165), (0.53, 0.165)]
    for k, (s, (x, y)) in enumerate(zip(ctx["samples"], pos)):
        a = _ease((t - 0.4 - 1.7 * k) / 0.6)
        if a <= 0: continue
        ax = f.add_axes([x, y + 0.075, 0.40, 0.255]); ax.imshow(s["img"], interpolation="bilinear", aspect="auto", alpha=a); ax.axis("off")
        g = _ease((t - 0.8 - 1.7 * k) / 1.0)
        bx = f.add_axes([x, y + 0.035, 0.40, 0.022]); bx.set_xlim(0, 100); bx.axis("off")
        bx.barh(0, 100, color=PANEL, height=1); bx.barh(0, s["pred"] * g, color=ACC if s["pred"] > 15 else TEAL, height=1)
        f.text(x, y + 0.005, f"Model {s['pred']*g:.0f}%   ·   Measured {s['true']:.0f}%", color=FG, fontsize=18, alpha=a)
    _chrome(f, ctx, f"ResNet18 fine-tuned on {ctx['n_images']:,} DeepSolarEye images · "
                    f"error {ctx['mae']:.1f} pts on days it never saw")

def _months(ax, dates):
    idx = [i for i, d in enumerate(dates) if d.day == 1]
    ax.set_xticks(idx); ax.set_xticklabels([dates[i].strftime("%b")[0] for i in idx])

def _scene_irr(f, t, ctx):
    ghi, rain, dates = ctx["ghi"], ctx["rain"], ctx["dates"]
    n = len(ghi); k = max(2, int(_ease(t / 7.0) * n))
    f.text(0.5, 0.885, f"Sunlight at {ctx['site_short']} · {ctx['year']}", ha="center", color=FG, fontsize=32, fontweight="bold")
    f.text(0.5, 0.80, f"{ctx['ghi_avg'] * _ease(t / 7.0):.1f} kWh/m² per day", ha="center", color=ACC, fontsize=46, fontweight="bold")
    ax = f.add_axes([0.10, 0.20, 0.82, 0.52]); _ax_style(ax)
    ax.set_xlim(0, n); ax.set_ylim(0, max(ghi) * 1.15)
    roll = pd.Series(ghi).rolling(7, center=True, min_periods=1).mean().values
    ax.plot(np.arange(k), ghi[:k], color=TEAL, lw=1.2, alpha=0.45)
    ax.plot(np.arange(k), roll[:k], color=ACC, lw=4)
    ax.set_ylabel("kWh/m²/day", color=MUTED, fontsize=15); _months(ax, dates)
    ax2 = ax.twinx(); ax2.set_ylim(max(rain.max(), 1) * 3, 0); ax2.axis("off")
    ax2.bar(np.arange(k), rain[:k], color="#4EA8FF", alpha=0.8, width=1.0)
    f.text(0.92, 0.735, "rain ↓", color="#4EA8FF", fontsize=16, ha="right")
    _chrome(f, ctx, f"NASA POWER daily irradiance (orange) and rain (blue) · {ctx['rain_days']} days washed by heavy rain")

def _scene_soil(f, t, ctx):
    curves = ctx["soil_curves"]; dates = ctx["dates"]; n = len(dates); k = max(2, int(_ease(t / 8.0) * n))
    f.text(0.5, 0.885, "Dust builds up. Rain and washing reset it.", ha="center", color=FG, fontsize=32, fontweight="bold")
    ax = f.add_axes([0.10, 0.22, 0.82, 0.55]); _ax_style(ax)
    ymax = max(c.max() for c, _ in curves.values()) * 100 * 1.15 + 1
    ax.set_xlim(0, n); ax.set_ylim(0, ymax); ax.set_ylabel("power lost to dust (%)", color=MUTED, fontsize=15); _months(ax, dates)
    for j, (lab, (c, col)) in enumerate(curves.items()):
        ax.plot(np.arange(k), c[:k] * 100, color=col, lw=3 if j else 2.5, alpha=0.95)
        f.text(0.12 + 0.28 * j, 0.80, f"● {lab}", color=col, fontsize=18)
    _chrome(f, ctx, f"Dust adds ~{ctx['soiling_rate']}%/day (field rate measured in Gandhinagar); "
                    f"≥{ctx['rain_clean_mm']:g} mm of rain or a wash resets it")

def _scene_money(f, t, ctx):
    v = ctx["loss_uncleaned_cr"] * _ease((t - 0.3) / 4.0)
    f.text(0.5, 0.80, "If a 1 GW park in Kutch is never washed, dust costs", ha="center", color=FG, fontsize=27)
    f.text(0.5, 0.635, f"₹{v:,.0f} crore", ha="center", color=ACC, fontsize=112, fontweight="bold")
    f.text(0.5, 0.535, "a year (estimate)", ha="center", color=FG, fontsize=27)
    a = _ease((t - 5.0) / 0.8)
    if a > 0:
        f.text(0.5, 0.40, f"Washing every {ctx['headline_interval']} days wins back", ha="center", color=FG, fontsize=30, alpha=a)
        f.text(0.5, 0.305, f"₹{ctx['recovered_cr']:,.0f} crore / year", ha="center", color=TEAL, fontsize=60, fontweight="bold", alpha=a)
    f.text(0.5, 0.19, "\n".join(_tw.wrap(ctx["how_line"], 70)), ha="center", va="center", color=MUTED, fontsize=17,
           alpha=_ease((t - 6.0) / 0.8))
    _chrome(f, ctx, "Independent study on public data · every assumption is a variable in the notebook")

def _scene_end(f, t, ctx):
    a = _ease(t / 0.8)
    f.text(0.5, 0.60, "23N AI Labs", ha="center", color=FG, fontsize=76, fontweight="bold", alpha=a)
    f.text(0.5, 0.50, f"Field Test #{ctx['ft_num']} · Open notebook in comments", ha="center", color=ACC, fontsize=30, alpha=a)
    f.text(0.5, 0.40, "23nlabs.com", ha="center", color=TEAL, fontsize=34, fontweight="bold", alpha=a)
    f.text(0.5, 0.12, "Data: DeepSolarEye (CC BY-NC-SA) · NASA POWER", ha="center", color=MUTED, fontsize=16, alpha=a)

SCENES = [  # (seconds, draw fn, seconds after which the frame is static)
    (5.0, _scene_hook, 1.5), (12.0, _scene_panels, 7.0), (11.0, _scene_irr, 7.1),
    (12.0, _scene_soil, 8.1), (12.0, _scene_money, 7.0), (6.0, _scene_end, 0.9)]

def render_video(ctx, out_mp4, fps=FPS):
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgba", "-s", f"{VW}x{VH}",
           "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "veryfast", "-pix_fmt", "yuv420p",
           "-crf", "20", "-movflags", "+faststart", out_mp4]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    f = _fig(); t0 = time.time(); nframes = 0
    for dur, fn, static_after in SCENES:
        buf = None
        for i in range(int(round(dur * fps))):
            t = i / fps
            if buf is None or t <= static_after:
                f.clear(); fn(f, t, ctx); f.canvas.draw(); buf = bytes(f.canvas.buffer_rgba())
            proc.stdin.write(buf); nframes += 1
    proc.stdin.close(); proc.wait()
    if proc.returncode: raise RuntimeError("ffmpeg failed")
    print(f"wrote {out_mp4}: {nframes} frames ({nframes/fps:.1f}s) in {time.time()-t0:.0f}s")
    return out_mp4

def render_cover(ctx, out_png):
    f = _fig(1080, 1350)
    f.text(0.05, 0.955, f"23N FIELD TESTS · #{ctx['ft_num']}", color=ACC, fontsize=19, fontweight="bold", va="top")
    f.text(0.05, 0.895, ctx["title"], color=FG, fontsize=58, fontweight="bold", va="top")
    f.text(0.05, 0.815, ctx["hook"], color=MUTED, fontsize=27, va="top")
    f.text(0.05, 0.725, f"₹{ctx['loss_uncleaned_cr']:,.0f} crore", color=ACC, fontsize=100, fontweight="bold", va="top")
    f.text(0.05, 0.615, "a year lost to dust by a 1 GW park in Kutch\nif it were never washed (estimate)", color=FG, fontsize=25, va="top", linespacing=1.3)
    f.text(0.05, 0.535, f"Washing every {ctx['headline_interval']} days wins back ₹{ctx['recovered_cr']:,.0f} crore",
           color=TEAL, fontsize=27, fontweight="bold", va="top")
    for k, s in enumerate(ctx["samples"][:4]):
        ax = f.add_axes([0.05 + k * 0.23, 0.285, 0.21, 0.21 * 1080 / 1350]); ax.imshow(s["img"]); ax.axis("off")
        f.text(0.05 + k * 0.23, 0.268, f"model {s['pred']:.0f}% loss", color=ACC if s["pred"] > 15 else TEAL, fontsize=19, va="top")
        f.text(0.05 + k * 0.23, 0.241, f"measured {s['true']:.0f}%", color=MUTED, fontsize=16, va="top")
    f.text(0.05, 0.125, "CNN on open panel images + NASA POWER sunlight\nIndependent study on public data",
           color=MUTED, fontsize=19, linespacing=1.4)
    f.text(0.05, 0.06, "23N AI Labs · 23nlabs.com", color=FG, fontsize=24, fontweight="bold")
    f.savefig(out_png, dpi=100, facecolor=BG); print("wrote", out_png); return out_png

In [ ]:
VIDEO_CTX = dict(
    ft_num=FT_NUM, title=FT_TITLE, hook="What does dust cost India's solar parks?",
    hook_sub=f"A 1 GW park · {SITE_NAME} · {YEAR} data",
    samples=SAMPLES, mae=MAE_PTS, n_images=N_TRAIN,
    dates=list(wx.index), ghi=GHI, rain=RAIN, ghi_avg=GHI_AVG, year=YEAR, site_short="Khavda, Kutch",
    rain_days=int((RAIN >= RAIN_CLEAN_MM).sum()), rain_clean_mm=RAIN_CLEAN_MM,
    soil_curves={"never washed": (SCEN[None]["s"], MUTED), f"every {SLOW} days": (SCEN[SLOW]["s"], ACC),
                 f"every {HEADLINE_INTERVAL} days": (SCEN[HEADLINE_INTERVAL]["s"], TEAL)},
    soiling_rate=SOILING_RATE_PCT_PER_DAY, loss_uncleaned_cr=LOSS_UNCLEANED_CR, recovered_cr=RECOVERED_CR,
    headline_interval=HEADLINE_INTERVAL,
    how_line=(f"How: 1 GW × NASA POWER sunlight × PR {PERFORMANCE_RATIO} × ₹{TARIFF_RS_PER_KWH:.2f}/kWh × "
              f"{SOILING_RATE_PCT_PER_DAY}%/day dust build-up, reset by ≥{RAIN_CLEAN_MM:g} mm rain or a wash"))
VIDEO = render_video(VIDEO_CTX, f"ft{FT_NUM}_video.mp4")
COVER = render_cover(VIDEO_CTX, f"ft{FT_NUM}_cover.png")
print(subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                      "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "compact", VIDEO],
                     capture_output=True, text=True).stdout)
try:
    from google.colab import files
    files.download(VIDEO); files.download(COVER)
except Exception:
    print("Not on Colab: find the files in the working directory (Kaggle: /kaggle/working).")

## 7. Post text (filled with this run's numbers — ready to paste)

In [ ]:
LINKEDIN_TEMPLATE = "What does dust cost India's solar parks?\n\n23N Field Tests #01 - The Solar Dust Tax\n\nLeft uncleaned, dust would cost a 1 GW solar park in Kutch, Gujarat ~₹{loss_uncleaned_cr} crore a year. Cleaning every {headline_interval} days wins back ~₹{recovered_cr} crore of that.\n\nHow we got there (independent study, public data only):\n- Vision: a ResNet18 fine-tuned on {n_train} open DeepSolarEye panel images predicts % power loss from a photo. Error: {mae_pts} percentage points (MAE) on days it never saw.\n- Sunlight: NASA POWER daily irradiance and rain at Khavda, Kutch for {year} (avg {ghi_avg} kWh/m2/day).\n- Dust: {soiling_rate}%/day build-up (a field rate measured in Gandhinagar), reset by heavy rain or a wash.\n- Money: 1 GW x PR {pr} x ₹{tariff}/kWh.\n\nStill lost per year when cleaning every {interval_list} days: ₹{interval_losses} crore.\nGoing from every {slow_interval} to every {headline_interval} days pays off if a wash costs under ~₹{breakeven_rs_per_mw} per MW.\n\nThese are estimates. Every assumption is a named variable in the notebook - change the tariff, dust rate or site and re-run.\n\nNotebook + live demo (upload a panel photo, get ₹ lost per MW) in the comments.\n\n#SolarEnergy #AI #Gujarat #RenewableEnergy #23NFieldTests"
X_TEMPLATE = 'Dust costs a 1 GW solar park in Kutch ~₹{loss_uncleaned_cr} crore a year if left uncleaned. Washing every {headline_interval} days wins back ~₹{recovered_cr} crore.\n\nCNN on open panel images + NASA sunlight. Independent study, open notebook.\n\n23N Field Tests #01'
X_THREAD = [
    "1/ What does dust cost India's solar parks? We modelled a 1 GW park in Kutch on public data: ~₹{loss_uncleaned_cr} crore/yr lost if never washed; ~₹{recovered_cr} crore won back by washing every {headline_interval} days. #23NFieldTests",
    '2/ A ResNet18 trained on {n_train} DeepSolarEye images reads % power loss from a panel photo ({mae_pts} pts MAE on unseen days). NASA POWER gives {year} sunlight + rain at Khavda ({ghi_avg} kWh/m2/day).',
    '3/ Dust adds {soiling_rate}%/day (Gandhinagar field rate); heavy rain or a wash resets it. Tariff ₹{tariff}/kWh. All assumptions are variables - notebook + demo in replies.']

POST_VARS = dict(
    loss_uncleaned_cr=f"{LOSS_UNCLEANED_CR:,.0f}", recovered_cr=f"{RECOVERED_CR:,.0f}", headline_interval=HEADLINE_INTERVAL,
    n_images=f"{N_IMAGES:,}", n_train=f"{N_TRAIN:,}", mae_pts=f"{MAE_PTS:.1f}", year=YEAR, ghi_avg=f"{GHI_AVG:.1f}",
    soiling_rate=SOILING_RATE_PCT_PER_DAY, pr=PERFORMANCE_RATIO, tariff=f"{TARIFF_RS_PER_KWH:.2f}",
    interval_list=" / ".join(str(n) for n in CLEANING_INTERVALS),
    interval_losses=" / ".join(f"{SCEN[n]['lost_cr']:,.0f}" for n in CLEANING_INTERVALS),
    slow_interval=SLOW, breakeven_rs_per_mw=f"{BREAKEVEN_RS_PER_MW:,.0f}")

li = LINKEDIN_TEMPLATE.format(**POST_VARS)
xp = X_TEMPLATE.format(**POST_VARS)
thread = [t.format(**POST_VARS) for t in X_THREAD]
print("=" * 30, f"LinkedIn ({len(li)} chars, limit 1300)", "=" * 30); print(li)
print("=" * 30, f"X ({len(xp)} chars, limit 280)", "=" * 30); print(xp)
for t in thread: print("-" * 10, len(t), "chars"); print(t)
assert len(li) <= 1300 and len(xp) <= 280 and all(len(t) <= 280 for t in thread), "post too long — edit template"
with open(f"ft{FT_NUM}_posts_filled.txt", "w") as fh:
    fh.write(li + "\n\n" + "=" * 40 + "\n" + xp + "\n\n" + "\n\n".join(thread))

## 8. Limitations (read before quoting the number)
1. **The CNN and the money model are separate.** The headline ₹ figure comes from the soiling-rate model, not from
   the CNN. The CNN shows that loss can be *measured* from cheap camera images (useful for deciding *when* to wash);
   it was not run on Kutch imagery, because no open labelled imagery of Kutch panels exists.
2. **DeepSolarEye is one rooftop rig, ~2 weeks, one panel type** with mostly artificial, patchy soiling (powders,
   leaves), not the uniform fine dust of the Rann. The model will not generalise to drone/satellite views or other
   panels without fine-tuning. Our MAE is on held-out *days* from the same rig — the realistic error on a new site
   is higher.
3. **The soiling rate (0.37 %/day) was measured in Gandhinagar**, ~300 km from Khavda, and the paper reports that
   light rain and humidity can double it. The Rann is also saltier and windier. The sensitivity table shows how the
   result moves at 0.2–0.5 %/day; site-specific soiling stations would replace this assumption.
4. **"Never washed" is a counterfactual.** Real parks already wash on some schedule, so the money actually at stake
   for an operator is the *difference between schedules*, not the uncleaned total.
5. **Simple yield model:** horizontal irradiance × one performance ratio. No tilt/trackers (would raise yield and
   losses), no temperature model, no curtailment or clipping, no degradation. NASA POWER is satellite/reanalysis
   data on a ~50 km grid, not a ground pyranometer.
6. **Tariff** is a single flat number from recent GUVNL auctions; actual PPAs at any specific park differ.
7. **Washing cost is not modelled** — we report the break-even ₹ per MW per wash instead. Water use, labour, robot
   capex and module wear from frequent washing all matter in practice.
8. **Licence:** DeepSolarEye is CC BY-NC-SA 2.0 (non-commercial, share-alike). The trained weights inherit those
   terms; do not use them commercially.